In [1]:
# Paso 0: Importar librerias

import os
import json
import gzip
import pickle as pkl
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, recall_score, f1_score, precision_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA

In [2]:
# Paso 1: Limpiar datasets

def clean_dataset(df):
    df = df.rename(columns={"default payment next month": "default"})
    df = df.drop("ID", axis=1)
    df["EDUCATION"] = df["EDUCATION"].apply(lambda x: x if x in range(5) else 4)
    df.dropna()
    return df

data_train = pd.read_csv("../files/input/train_data.csv.zip", compression="zip")
data_test = pd.read_csv("../files/input/test_data.csv.zip", compression="zip")
data_train = clean_dataset(data_train)
data_test = clean_dataset(data_test)
data_train.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,310000,1,3,1,32,0,0,0,0,0,...,84373,57779,14163,8295,6000,4000,3000,1000,2000,0
1,10000,2,3,1,49,-1,-1,-2,-1,2,...,1690,1138,930,0,0,2828,0,182,0,1
2,50000,1,2,1,28,-1,-1,-1,0,-1,...,45975,1300,43987,0,46257,2200,1300,43987,1386,0
3,80000,2,3,1,52,2,2,3,3,3,...,40748,39816,40607,3700,1600,1600,0,1600,1600,1
4,270000,1,1,2,34,1,2,0,0,2,...,22448,15490,17343,0,4000,2000,0,2000,2000,0


In [3]:
# Paso 2: dividir datasets

X_train = data_train.drop("default", axis=1)
y_train = data_train["default"]
X_test = data_test.drop("default", axis=1)
y_test = data_test["default"]

In [4]:
# Paso 3: Crear el pipeline

cat_cols = ["SEX", "MARRIAGE", "EDUCATION"]
num_cols = [col for col in X_train.columns if col not in cat_cols]
prep = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
    ])
pipe = Pipeline(steps=[
    ("preprocessor", prep),
    ("pca", PCA()),
    ("f_sel", SelectKBest(f_classif)),
    ("model", SVC())
])

In [ ]:
# Paso 4: Ajustar hiperparámetros

params = {
    'f_sel__k': [5, 10, 15, 20, 25],
    'model__C': [0.1, 1, 10],
    'model__kernel': ['linear', 'rbf', 'sigmoid'],
}

grid = GridSearchCV(pipe, param_grid=params, cv=10, n_jobs=-1, scoring="balanced_accuracy")
grid.fit(X_train, y_train)

In [ ]:
# Paso 5: Guardar el modelo

os.makedirs("../files/models", exist_ok=True)
with gzip.open("../files/models/model.pkl.gz", "wb") as f:
    pkl.dump(grid, f)

In [ ]:
# Paso 6: Calcular métricas del modelo

def metrics_dicc(y_true, y_pred, df_name):
    dicc = {
        "type" : "metrics",
        "dataset" : df_name,
        "precision" : precision_score(y_true, y_pred),
        "balanced_accuracy" : balanced_accuracy_score(y_true, y_pred),
        "recall" : recall_score(y_true, y_pred),
        "f1_score" : f1_score(y_true, y_pred)
    }
    return dicc

def confusion_to_dict(y_true, y_pred, df_name):
    cm = confusion_matrix(y_true, y_pred)
    return {
        "type": "cm_matrix",
        "dataset": df_name,
        "true_0": {"predicted_0": int(cm[0, 0]), "predicted_1": int(cm[0, 1])},
        "true_1": {"predicted_0": int(cm[1, 0]), "predicted_1": int(cm[1, 1])}
    }

y_train_pred = grid.predict(X_train)
y_test_pred = grid.predict(X_test)
metrics = [metrics_dicc(y_train, y_train_pred, "train"), metrics_dicc(y_test, y_test_pred, "test"),
           confusion_to_dict(y_train, y_train_pred, "train"), confusion_to_dict(y_test, y_test_pred, "test")]
os.makedirs("../files/output", exist_ok=True)
with open("../files/output/metrics.json", "w") as file:
    for row in metrics:
        json.dump(row, file)
        file.write("\n")